# flask API

## Objective

The objective of this notebook is to create a Flask API that loads the trained churn prediction model and provides predictions for new customer data.


### Why Flask?

Flask is a lightweight python web framework that can be used to build APIs. In this project, Flask will provide an endpoint that receives customer information and returns a churn prediction.


In [155]:
# Import pandas for data manipulation
import pandas as pd

# Load encoded dataset

df=pd.read_csv("../data/processed/encoded_telco_churn.csv")
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TenureGroup_0-12,TenureGroup_13-24,TenureGroup_25-48,TenureGroup_49-72,MonthlyChargeCategory_High,MonthlyChargeCategory_Low,MonthlyChargeCategory_Medium
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0,1,0,1,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0,0,1,0,0,1,0,0,0,1
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0,0,1,1,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,0,0,0,0,0,1,0,0,0,1
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0,1,0,1,0,0,0,1,0,0


In [156]:
# Seperate input features and target variables

x=df.drop("Churn",axis=1)
y=df["Churn"]

In [157]:
# display feature names used by the model

print(x.columns.tolist())

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'TotalServices', 'LongTermCustomer', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed

In [158]:
# Check how many features model expects

print("Number of features:",x.shape[1])

Number of features: 49


### Understand model input

The model was trained using the encoded dataset, so the API must provide input features in the same structure used during training. Checking the features helps prevent incorrect input from being passed to the model. 

### Observation

The encoded datset contains the features used during model training. The API must maintain the same feature structure and order when making predictions for new customers.

In [159]:
# Impot os to inspect the models directory

import os

# Display files stored in the models folder

print(os.listdir("../models"))

['tuned_random_forest.pkl']


In [160]:
# Import joblib for loading trained model

import joblib
model = joblib.load("../models/tuned_random_forest.pkl")
print(model)


RandomForestClassifier(max_depth=10, min_samples_leaf=2, min_samples_split=5,
                       n_estimators=300, random_state=42)


### Observation

The Trained Random Forest model was successfully loaded and is ready to make predictions.

## Test Model Prediction

Before integrating the model with Flask, a sample prediction is performed to verify that the saved model can correctly accept the encoded feature structure and return a churn prediction.


In [161]:
# Select one customer from the encoded feature dataset

sample = x.iloc[[0]]
sample

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,TotalServices,...,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TenureGroup_0-12,TenureGroup_13-24,TenureGroup_25-48,TenureGroup_49-72,MonthlyChargeCategory_High,MonthlyChargeCategory_Low,MonthlyChargeCategory_Medium
0,0,0,1,0,1,0,1,29.85,29.85,2,...,0,1,0,1,0,0,0,0,1,0


In [162]:
# Generate a churn prediction for the sample customer

sample_pred = model.predict(sample)
print("Churn Prediction:",sample_pred[0])

Churn Prediction: 1


In [163]:
# Get probabilty of each class

sample_prob =model.predict_proba(sample)

print("Probabilty of No Churn:", sample_prob[0][0])
print("Probability of Churn:", sample_prob[0][1])


Probabilty of No Churn: 0.499620513018118
Probability of Churn: 0.5003794869818818


### Observation

The trained Random Forest Model successfully genereated a churn prediction for a sample customer. The model also provided the probability associated with each class, which gives additional information about the prediction.

In [164]:
# Select another customer

sample2 = x.iloc[[1]]
sample_pred2 = model.predict(sample2)
print("Churn prediction:", sample_pred2[0])

Churn prediction: 0


In [165]:
# Display churn probabilty

sample_prob2 = model.predict_proba(sample2)

print("Probability of No Churn:", sample_prob2[0][0])
print("Probabilty of Churn:", sample_prob2[0][1])

Probability of No Churn: 0.9776390380133936
Probabilty of Churn: 0.022360961986606098


### Create Flask API

The Flask API will provide an endpoint that accepts customer data and returns a churn prediction from the trained machine learning model.

In [166]:
# Import Flask commands for creating an API

from flask import Flask, request ,jsonify

# create Flask application

app = Flask(__name__)

In [167]:
# Define the home endpoint

@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message":"Telecom Customer Churn Prediction API is running"
    })

## Prediction Endpoint

The '/predict' endpoint will receive customer information and return the predicted churn class.

In [168]:
# Define the prediction endpoint

@app.route("/predict",methods=["POST"])
def predict():

    # Get customer data sent as JSON
    data = request.get_json()

    # Convert the input data into a DataFrame
    customer_data = pd.DataFrame([data])

    # Generate churn prediction
    prediction = model.predict(customer_data)

    # Generate churn probability
    probability = model.predict_proba(customer_data)

    # Return the prediction and probabilties as JSON
    return jsonify({
        "churn_prediction": int(prediction[0]),
        "probability_no_churn": float(probability[0][0]),
        "probability_churn": float(probability[0][1])
     }) 

In [169]:
# Display the number of features expected

print("Expected features:", len(x.columns))

Expected features: 49


<!-- # Test API input
 
The API must receive the same features and feature order used during model training. A sample input is created from the encoded feature structure to test the prediction endpoint. -->


## Test API Input

The API must receive the same features and feature order used during model training. A sample input is created from the encoded feature structure to test the prediction endpoint.

In [170]:
# Select one customer from the encoded dataset

test_customer = x.iloc[0].to_dict()

# Display the test input

print(test_customer)

{'gender': 0.0, 'SeniorCitizen': 0.0, 'Partner': 1.0, 'Dependents': 0.0, 'tenure': 1.0, 'PhoneService': 0.0, 'PaperlessBilling': 1.0, 'MonthlyCharges': 29.85, 'TotalCharges': 29.85, 'TotalServices': 2.0, 'LongTermCustomer': 0.0, 'MultipleLines_No': 0.0, 'MultipleLines_No phone service': 1.0, 'MultipleLines_Yes': 0.0, 'InternetService_DSL': 1.0, 'InternetService_Fiber optic': 0.0, 'InternetService_No': 0.0, 'OnlineSecurity_No': 1.0, 'OnlineSecurity_No internet service': 0.0, 'OnlineSecurity_Yes': 0.0, 'OnlineBackup_No': 0.0, 'OnlineBackup_No internet service': 0.0, 'OnlineBackup_Yes': 1.0, 'DeviceProtection_No': 1.0, 'DeviceProtection_No internet service': 0.0, 'DeviceProtection_Yes': 0.0, 'TechSupport_No': 1.0, 'TechSupport_No internet service': 0.0, 'TechSupport_Yes': 0.0, 'StreamingTV_No': 1.0, 'StreamingTV_No internet service': 0.0, 'StreamingTV_Yes': 0.0, 'StreamingMovies_No': 1.0, 'StreamingMovies_No internet service': 0.0, 'StreamingMovies_Yes': 0.0, 'Contract_Month-to-month': 1.

In [171]:
import json

# Convert the sample customer dictonary to JSON format
test_json = json.dumps(test_customer)

print(test_json)

{"gender": 0.0, "SeniorCitizen": 0.0, "Partner": 1.0, "Dependents": 0.0, "tenure": 1.0, "PhoneService": 0.0, "PaperlessBilling": 1.0, "MonthlyCharges": 29.85, "TotalCharges": 29.85, "TotalServices": 2.0, "LongTermCustomer": 0.0, "MultipleLines_No": 0.0, "MultipleLines_No phone service": 1.0, "MultipleLines_Yes": 0.0, "InternetService_DSL": 1.0, "InternetService_Fiber optic": 0.0, "InternetService_No": 0.0, "OnlineSecurity_No": 1.0, "OnlineSecurity_No internet service": 0.0, "OnlineSecurity_Yes": 0.0, "OnlineBackup_No": 0.0, "OnlineBackup_No internet service": 0.0, "OnlineBackup_Yes": 1.0, "DeviceProtection_No": 1.0, "DeviceProtection_No internet service": 0.0, "DeviceProtection_Yes": 0.0, "TechSupport_No": 1.0, "TechSupport_No internet service": 0.0, "TechSupport_Yes": 0.0, "StreamingTV_No": 1.0, "StreamingTV_No internet service": 0.0, "StreamingTV_Yes": 0.0, "StreamingMovies_No": 1.0, "StreamingMovies_No internet service": 0.0, "StreamingMovies_Yes": 0.0, "Contract_Month-to-month": 1.

In [172]:
# Convert the test customer back into DataFrame

test_df = pd.DataFrame([test_customer])

# Generate prediction

prediction = model.predict(test_df)

# Generate prediction probabilities

probability = model.predict_proba(test_df)

print("Churn prediction:", prediction[0])
print("Probability of No Churn:", probability[0][0])
print("Probability of Churn:", probability[0][1])

Churn prediction: 1
Probability of No Churn: 0.499620513018118
Probability of Churn: 0.5003794869818818


<!-- 
### Observation

The model predicted that the sample customer is likely to churn. However, the prediction probabilities are almost evenly divided between the two classes, with approximately 49.96% probability of no churn and 50.04% probability of churn.

This indicates that the model has low confidence in this particular prediction. -->


### Observation

The sample customer data was successfully converted into the feature structure expected by the trained model. The model generated both a churn prediction and the probability of each class, confirming that the test input is suitable for the API.

## Testing the Flask Application

The Flask application is implemented separately in `app/app.py`. The notebook is used to send requests to the running API and verify its responses.

In [173]:
# Run the Flask application
# if __name__ == "__main__":
#     app.run(debug=False,use_reloader= False)

In [174]:
# Import requests for sending HTTP requests
import requests

In [175]:
# Send the GET request to the Flask home endpoint.

response = requests.get("http://127.0.0.1:5000/")

# Display API response
print(response.json())

{'message': 'Telecom Customer Churn Prediction API is running'}


### Observation

The Flask application successfully responded to a GET request through the home endpoint. This confirms that the application is running correctly and can be accessed locally.

## Test Churn Prediction API

In [176]:
# Send a test request to the prediction endpoint

response = requests.post("http://127.0.0.1:5000/predict",
json={}
)

# Display the response

print(response.status_code)
print(response.text)

500
<!doctype html>
<html lang=en>
  <head>
    <title>KeyError: &#34;None of [Index([&#39;gender&#39;, &#39;SeniorCitizen&#39;, &#39;Partner&#39;, &#39;Dependents&#39;, &#39;tenure&#39;,\n       &#39;PhoneService&#39;, &#39;PaperlessBilling&#39;, &#39;MonthlyCharges&#39;, &#39;TotalCharges&#39;,\n       &#39;TotalServices&#39;, &#39;LongTermCustomer&#39;, &#39;MultipleLines_No&#39;,\n       &#39;MultipleLines_No phone service&#39;, &#39;MultipleLines_Yes&#39;,\n       &#39;InternetService_DSL&#39;, &#39;InternetService_Fiber optic&#39;,\n       &#39;InternetService_No&#39;, &#39;OnlineSecurity_No&#39;,\n       &#39;OnlineSecurity_No internet service&#39;, &#39;OnlineSecurity_Yes&#39;,\n       &#39;OnlineBackup_No&#39;, &#39;OnlineBackup_No internet service&#39;,\n       &#39;OnlineBackup_Yes&#39;, &#39;DeviceProtection_No&#39;,\n       &#39;DeviceProtection_No internet service&#39;, &#39;DeviceProtection_Yes&#39;,\n       &#39;TechSupport_No&#39;, &#39;TechSupport_No internet service&

### Check Model Input Features



In [177]:
# Load the trained model

import joblib

model= joblib.load("../models/tuned_random_forest.pkl")

# Display the features used by the trained model

model.feature_names_in_

# Check no of features expected by the model

print("Numbr of features:", len(model.feature_names_in_))\

SyntaxError: incomplete input (1392090409.py, line 13)

In [ ]:
# Display  all features expected by the trained model

for i, feature in enumerate(model.feature_names_in_, start=1):
    print(i,feature)

1 gender
2 SeniorCitizen
3 Partner
4 Dependents
5 tenure
6 PhoneService
7 PaperlessBilling
8 MonthlyCharges
9 TotalCharges
10 TotalServices
11 LongTermCustomer
12 MultipleLines_No
13 MultipleLines_No phone service
14 MultipleLines_Yes
15 InternetService_DSL
16 InternetService_Fiber optic
17 InternetService_No
18 OnlineSecurity_No
19 OnlineSecurity_No internet service
20 OnlineSecurity_Yes
21 OnlineBackup_No
22 OnlineBackup_No internet service
23 OnlineBackup_Yes
24 DeviceProtection_No
25 DeviceProtection_No internet service
26 DeviceProtection_Yes
27 TechSupport_No
28 TechSupport_No internet service
29 TechSupport_Yes
30 StreamingTV_No
31 StreamingTV_No internet service
32 StreamingTV_Yes
33 StreamingMovies_No
34 StreamingMovies_No internet service
35 StreamingMovies_Yes
36 Contract_Month-to-month
37 Contract_One year
38 Contract_Two year
39 PaymentMethod_Bank transfer (automatic)
40 PaymentMethod_Credit card (automatic)
41 PaymentMethod_Electronic check
42 PaymentMethod_Mailed check
4

In [ ]:
# Create a test customer using the 49 features expected by the model

sample = pd.DataFrame([{
    "gender": 1,
    "SeniorCitizen": 0,
    "Partner": 1,
    "Dependents": 0,
    "tenure": 12,
    "PhoneService": 1,
    "PaperlessBilling": 1,
    "MonthlyCharges": 70.0,
    "TotalCharges": 840.0,
    "TotalServices": 3,
    "LongTermCustomer": 0,

    "MultipleLines_No": 1,
    "MultipleLines_No phone service": 0,
    "MultipleLines_Yes": 0,

    "InternetService_DSL": 1,
    "InternetService_Fiber optic": 0,
    "InternetService_No": 0,

    "OnlineSecurity_No": 1,
    "OnlineSecurity_No internet service": 0,
    "OnlineSecurity_Yes": 0,

    "OnlineBackup_No": 1,
    "OnlineBackup_No internet service": 0,
    "OnlineBackup_Yes": 0,

    "DeviceProtection_No": 1,
    "DeviceProtection_No internet service": 0,
    "DeviceProtection_Yes": 0,

    "TechSupport_No": 1,
    "TechSupport_No internet service": 0,
    "TechSupport_Yes": 0,

    "StreamingTV_No": 1,
    "StreamingTV_No internet service": 0,
    "StreamingTV_Yes": 0,

    "StreamingMovies_No": 1,
    "StreamingMovies_No internet service": 0,
    "StreamingMovies_Yes": 0,

    "Contract_Month-to-month": 1,
    "Contract_One year": 0,
    "Contract_Two year": 0,

    "PaymentMethod_Bank transfer (automatic)": 0,
    "PaymentMethod_Credit card (automatic)": 0,
    "PaymentMethod_Electronic check": 1,
    "PaymentMethod_Mailed check": 0,

    "TenureGroup_0-12": 1,
    "TenureGroup_13-24": 0,
    "TenureGroup_25-48": 0,
    "TenureGroup_49-72": 0,

    "MonthlyChargeCategory_High": 0,
    "MonthlyChargeCategory_Low": 0,
    "MonthlyChargeCategory_Medium": 1
}])

print("Number of columns:", sample.shape[1])

Number of columns: 49


In [ ]:
print(sample.columns.tolist() == list(model.feature_names_in_))

True


In [ ]:
# Test model directly

prediction = model.predict(sample)
probability = model.predict_proba(sample)

print("Churn prediction:", prediction[0])
print("Probability of no churn", probability[0][0])
print("Probability of churn:", probability[0][1])

Churn prediction: 0
Probability of no churn 0.7250018191217174
Probability of churn: 0.2749981808782822


### Test Prediction Endpoint with Customer Data

A sample customer containing the same 49 features expected by the trained model is sent to the Flask '/predict' endpoint. The  returned prediction and probabilities are compared with direct model predictions.

In [ ]:
#print(type(model))
# print(model.n_features_in_)
# print(model.feature_names_in_)

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
49
['gender' 'SeniorCitizen' 'Partner' 'Dependents' 'tenure' 'PhoneService'
 'PaperlessBilling' 'MonthlyCharges' 'TotalCharges' 'TotalServices'
 'LongTermCustomer' 'MultipleLines_No' 'MultipleLines_No phone service'
 'MultipleLines_Yes' 'InternetService_DSL' 'InternetService_Fiber optic'
 'InternetService_No' 'OnlineSecurity_No'
 'OnlineSecurity_No internet service' 'OnlineSecurity_Yes'
 'OnlineBackup_No' 'OnlineBackup_No internet service' 'OnlineBackup_Yes'
 'DeviceProtection_No' 'DeviceProtection_No internet service'
 'DeviceProtection_Yes' 'TechSupport_No' 'TechSupport_No internet service'
 'TechSupport_Yes' 'StreamingTV_No' 'StreamingTV_No internet service'
 'StreamingTV_Yes' 'StreamingMovies_No'
 'StreamingMovies_No internet service' 'StreamingMovies_Yes'
 'Contract_Month-to-month' 'Contract_One year' 'Contract_Two year'
 'PaymentMethod_Bank transfer (automatic)'
 'PaymentMethod_Credit card (automatic)' 'PaymentMethod_Elect

In [ ]:
# Test the saved model directly

prediction = model.predict(sample)
probability = model.predict_proba(sample)

print("Churn prediction:", prediction[0])
print("Probability of no churn:", probability[0][0])
print("Probability of churn:", probability[0][1])

Churn prediction: 1
Probability of no churn: 0.499620513018118
Probability of churn: 0.5003794869818818


In [ ]:
# Convert the sampl customr to JSON
import requests
sample_json = sample.to_dict(orient="records")[0]

# Send the customer data to the Flask prediction endpoint
response = requests.post(
    "http://127.0.0.1:5000/predict",
    json = sample_json
)

# Display the API response

print("Status code:", response.status_code)
print("Response:", response.text)

Status code: 200
Response: {
  "Churn_prediction": 1,
  "Probability_churn": 0.5003794869818818,
  "Probability_no_churn": 0.499620513018118
}



In [178]:
print(sample_json)

{'gender': 0, 'SeniorCitizen': 0, 'Partner': 1, 'Dependents': 0, 'tenure': 1, 'PhoneService': 0, 'PaperlessBilling': 1, 'MonthlyCharges': 29.85, 'TotalCharges': 29.85, 'TotalServices': 2, 'LongTermCustomer': 0, 'MultipleLines_No': 0, 'MultipleLines_No phone service': 1, 'MultipleLines_Yes': 0, 'InternetService_DSL': 1, 'InternetService_Fiber optic': 0, 'InternetService_No': 0, 'OnlineSecurity_No': 1, 'OnlineSecurity_No internet service': 0, 'OnlineSecurity_Yes': 0, 'OnlineBackup_No': 0, 'OnlineBackup_No internet service': 0, 'OnlineBackup_Yes': 1, 'DeviceProtection_No': 1, 'DeviceProtection_No internet service': 0, 'DeviceProtection_Yes': 0, 'TechSupport_No': 1, 'TechSupport_No internet service': 0, 'TechSupport_Yes': 0, 'StreamingTV_No': 1, 'StreamingTV_No internet service': 0, 'StreamingTV_Yes': 0, 'StreamingMovies_No': 1, 'StreamingMovies_No internet service': 0, 'StreamingMovies_Yes': 0, 'Contract_Month-to-month': 1, 'Contract_One year': 0, 'Contract_Two year': 0, 'PaymentMethod_Ba

## Flask API Testing

The Flask API was tested by sending a sample customer record containing the 49 features expected by the trained Random Forest model. The API successfully processed the request and returned a churn prediction along with the probability of churn and no churn. 

The API returned a status code of 200, confirming that the prediction endpoint is functioning successfully.

### Observation

The prediction API successfully accepted the customer data and returned the predicted churn class and corresponding probabilities. For the test customer, the model predicted churn with a probability of approximately 50.04%, indicating a boarderline churn-risk case.